<a href="https://colab.research.google.com/github/AroojSaleem-995/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AroojSaleem-995/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
%pip -q install duckdb

In [5]:
import duckdb
from google.colab import userdata

con = duckdb.connect()

In [6]:
HF_TOKEN = userdata.get("HF_TOKEN")

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [7]:
REL = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/*.parquet'
)
"""

In [9]:
con.sql(f"""
DESCRIBE
SELECT *
FROM {REL}
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

One row represents one content item (content_hash_id) for one client (client_hash_id) on one reporting date (report_date).

Time Window:
March 2025 (2025-03-01 to 2025-03-31)

Prediction / Ranking Goal:
Rank content based on search performance to identify pages that should be prioritized for SEO optimization.

**Table(s) Used:**

I use the **fact_content_daily_performance** table from the **FlyRank/internship-warehouse** dataset. This table contains daily content performance metrics such as impressions, clicks, average position, sessions, and pageviews for each client-content combination. For this assignment, I use the partition **month = '2025-03'** to analyze a mid-panel month as recommended.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

1. gsc_impressions
   - Available at the decision time because search impressions are already recorded.

2. gsc_clicks
   - Available at the decision time because clicks have already occurred.

3. gsc_avg_position
   - Available before making optimization decisions because it reflects current search ranking.

4. ga4_pageviews
   - Available because pageview data has already been collected.

5. ga4_sessions
   - Available because session counts exist before deciding which content to improve.

### Target (Proxy)

Search performance ranking using CTR (Click Through Rate).

### Context Fields

- client_hash_id
- content_hash_id
- report_date

These identify the client, content item, and reporting date.

### Excluded

AI-generated recommendation columns and future outcome information are excluded because they could introduce data leakage.

In [29]:
feature_df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM {REL}
""").df()

feature_df

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,2025-03-01,client_73cda7b4e4f265ea,content_30744f0df03fae4d,2,0,65.500000,0,0
1,2025-03-01,client_73cda7b4e4f265ea,content_eb59ee4f00886795,6,0,74.000000,0,0
2,2025-03-01,client_73cda7b4e4f265ea,content_e487f64cb6f0bf09,1,0,72.000000,0,0
3,2025-03-01,client_73cda7b4e4f265ea,content_96977940a4c4b65e,7,0,8.428571,0,0
4,2025-03-01,client_73cda7b4e4f265ea,content_df939e2e25c9b557,2,0,41.000000,0,0
...,...,...,...,...,...,...,...,...
167854,2025-03-31,client_fef1a8f436438636,content_4ffab4f5dbcf220a,2,0,8.500000,0,0
167855,2025-03-31,client_fef1a8f436438636,content_e2139b422bb5c51d,1,0,7.000000,0,0
167856,2025-03-31,client_fef1a8f436438636,content_d212425025bbbd51,1,0,7.000000,0,0
167857,2025-03-31,client_fef1a8f436438636,content_7701feb12469085d,156,0,48.314103,0,0


In [30]:
feature_df.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions'],
      dtype='object')

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {REL}
""").df()

,rows,first_date,last_date
0,167859,2025-03-01,2025-03-31


In [19]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(
        CAST(client_hash_id AS STRING),
        CAST(content_hash_id AS STRING),
        CAST(report_date AS STRING)
    )) AS unique_rows
FROM {REL}
""").df()

,total_rows,unique_rows
0,167859,167859


In [12]:
con.sql(f"""
SELECT
COUNT(*) AS available_rows
FROM {REL}
WHERE gsc_data_available IS TRUE
""").df()

,available_rows
0,167859


## 4. Leakage Experiment



In [40]:
feature_df["high_clicks"] = (
    feature_df["gsc_clicks"] > 0
).astype(int)

In [47]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
X = feature_df[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions"
    ]
]

y = feature_df["high_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = DecisionTreeClassifier(random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Honest Accuracy:", accuracy_score(y_test, pred))

Honest Accuracy: 0.8966400571905159


In [48]:
X_leak = X.copy()

X_leak["gsc_clicks"] = feature_df["gsc_clicks"]

In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42
)

model = DecisionTreeClassifier(random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy with Leakage:", accuracy_score(y_test, pred))

Accuracy with Leakage: 1.0




### Leakage Experiment Results

The honest model achieved an accuracy of approximately **89.7%** using only features that would be available at prediction time.

After intentionally adding **gsc_clicks** (the feature used to create the label `high_clicks`) back into the input features, the model's accuracy increased to **100%**.

This demonstrates **label leakage** because the model was given information directly related to the target, producing an unrealistically perfect result. Removing the leaked feature restores a more realistic evaluation.

## 5. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This notebook only uses the March 2025 partition of the warehouse.

The results may not represent seasonal trends or longer-term content performance.

The analysis also relies on anonymized identifiers, so page-specific context is unavailable.

## Self-check

✔ Defined the unit of analysis

✔ Identified features, label, context and excluded fields

✔ Verified the data using three queries

✔ Explained the data limitations